# Tabelas hash

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Explicar como uma tabela hash acha uma chave em cerca de O(1): códigos hash, baldes, colisões e redimensionamento
- Escolher uma chave que junta coisas equivalentes, como as letras ordenadas de um anagrama
- Trocar um laço aninhado por um mapa de contagens ou de somas de prefixo
- Construir uma tabela hash à mão

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Uma **tabela hash** acha uma chave mais ou menos no mesmo tempo, seja quantas chaves ela guardar. Uma **função hash** transforma a chave num número, seu **código hash**, e a tabela o usa para escolher um **balde**, como `índice = hash % baldes`.

![Quatro chaves em cinco baldes, com uma colisão no balde 2](buckets.pt-br.svg)

Duas chaves podem cair no mesmo balde: uma **colisão**. Cada balde guarda uma lista curta (encadeamento separado), ou a tabela tenta a próxima posição livre (endereçamento aberto). Quando a tabela fica cheia demais, ela dobra os baldes e reinsere tudo, o que mantém as listas curtas: O(1) por operação, em média.

#### 🎞️ Colocando chaves nos baldes

![As chaves 12, 7, 15 e 9 vão para o balde chave % 5; 12 e 7 colidem no balde 2, e uma busca só procura nesse balde](bucket-steps.pt-br.svg)

**Colisões acontecem:** duas strings diferentes com o mesmo código hash, e 1.000 palavras espalhadas em 16 baldes.

In [ ]:
IO.println("\"Aa\".hashCode() = " + "Aa".hashCode());
IO.println("\"BB\".hashCode() = " + "BB".hashCode());

int[] baldes = new int[16];
Random random = new Random(4);
for (int i = 0; i < 1000; i++) {
    String palavra = "palavra" + random.nextInt(1_000_000);
    baldes[Math.floorMod(palavra.hashCode(), 16)]++;
}
IO.println("palavras por balde: " + Arrays.toString(baldes));

Em Java, `HashMap` e `HashSet` são tabelas hash. Eles chamam o `hashCode()` da chave para achar o balde e o `equals()` para comparar dentro dele, então os dois precisam concordar, e uma chave não pode mudar enquanto está no mapa.

**equals sem hashCode:** a classe guarda os dois pontos "iguais"; o record, que escreve os dois métodos para você, guarda um.

In [ ]:
class PontoClasse {
    final int x;
    final int y;

    PontoClasse(int x, int y) {
        this.x = x;
        this.y = y;
    }

    @Override
    public boolean equals(Object o) {
        return o instanceof PontoClasse p && p.x == x && p.y == y;
    }
    // sem hashCode: pontos iguais normalmente caem em baldes diferentes
}

record PontoRecord(int x, int y) {}

void main() {
    Set<PontoClasse> classes = new HashSet<>(List.of(new PontoClasse(1, 2), new PontoClasse(1, 2)));
    Set<PontoRecord> records = new HashSet<>(List.of(new PontoRecord(1, 2), new PontoRecord(1, 2)));
    IO.println("classe sem hashCode: " + classes.size() + " pontos");
    IO.println("record:              " + records.size() + " ponto");
}

### ✏️ Exercício: Conte as letras

Monte um mapa de cada letra para quantas vezes ela aparece. `merge(chave, 1, Integer::sum)` soma um a uma contagem, começando em 1.

In [ ]:
Map<Character, Integer> letterCounts(String word) {
    // TODO: conte cada char de word num HashMap
    return new HashMap<>();
}

void main() {
    check("banana", letterCounts("banana"), Map.of('b', 1, 'a', 3, 'n', 2));
    check("vazio", letterCounts(""), Map.of());
    check("uma letra", letterCounts("zzz"), Map.of('z', 3));
}

<details>
<summary>Ver uma resposta</summary>

```java
Map<Character, Integer> letterCounts(String word) {
    Map<Character, Integer> counts = new HashMap<>();
    for (char c : word.toCharArray()) {
        counts.merge(c, 1, Integer::sum);
    }
    return counts;
}

void main() {
    check("banana", letterCounts("banana"), Map.of('b', 1, 'a', 3, 'n', 2));
    check("vazio", letterCounts(""), Map.of());
    check("uma letra", letterCounts("zzz"), Map.of('z', 3));
}
```

</details>

Tabelas hash substituem laços aninhados que perguntam "já vi isto?" ou "quantos?":

- **Contar:** `counts.merge(chave, 1, Integer::sum)`.
- **Agrupar** por uma chave escolhida: palavras com as mesmas letras ordenadas são anagramas.
- **Somas de prefixo:** a soma de `a[i..j]` é `prefix[j + 1] − prefix[i]`, então um mapa das somas de prefixo vistas até agora acha todo intervalo com uma soma dada numa passada.

**Somas de prefixo:** depois de uma passada, a soma de qualquer intervalo é uma subtração.

In [ ]:
int[] a = {3, 4, -1, 2, 5, -3};
long[] prefix = new long[a.length + 1];
for (int i = 0; i < a.length; i++) {
    prefix[i + 1] = prefix[i] + a[i];
}
IO.println("prefixos: " + Arrays.toString(prefix));
int[][] intervalos = {{0, 2}, {1, 3}, {2, 5}};
for (int[] r : intervalos) {
    IO.println("a[" + r[0] + ".." + r[1] + "] soma " + (prefix[r[1] + 1] - prefix[r[0]]));
}

### ✏️ Exercício: A primeira repetição

Devolva o primeiro caractere que aparece pela segunda vez, lendo da esquerda para a direita, ou `-` se nenhum se repete. Um HashSet lembra o que você já viu.

In [ ]:
char firstRepeat(String s) {
    // TODO: guarde num HashSet os chars vistos até agora
    return '-';
}

void main() {
    check("abcab", firstRepeat("abcab"), 'a');
    check("abccba", firstRepeat("abccba"), 'c');
    check("sem repetição", firstRepeat("xyz"), '-');
}

<details>
<summary>Ver uma resposta</summary>

```java
char firstRepeat(String s) {
    Set<Character> seen = new HashSet<>();
    for (char c : s.toCharArray()) {
        if (!seen.add(c)) {
            return c;
        }
    }
    return '-';
}

void main() {
    check("abcab", firstRepeat("abcab"), 'a');
    check("abccba", firstRepeat("abccba"), 'c');
    check("sem repetição", firstRepeat("xyz"), '-');
}
```

</details>

Cuidado: uma função hash que manda muitas chaves para o mesmo balde transforma a tabela numa lista lenta.

**O que uma função hash ruim custa:** as mesmas 10.000 chaves, com um bom hashCode e com um que devolve 42 para tudo.

In [ ]:
record ChaveBoa(int id) {}

record ChaveRuim(int id) {
    @Override
    public int hashCode() {
        return 42; // todas as chaves no mesmo balde
    }
}

void main() {
    int n = 10_000;
    long inicio = System.nanoTime();
    Set<ChaveBoa> boas = new HashSet<>();
    for (int i = 0; i < n; i++) {
        boas.add(new ChaveBoa(i));
    }
    long meio = System.nanoTime();
    Set<ChaveRuim> ruins = new HashSet<>();
    for (int i = 0; i < n; i++) {
        ruins.add(new ChaveRuim(i));
    }
    long fim = System.nanoTime();
    IO.println("hashCode bom: " + (meio - inicio) / 1_000_000 + " ms");
    IO.println("hashCode 42:  " + (fim - meio) / 1_000_000 + " ms");
}

## Leituras

- [Hash Table](https://visualgo.net/en/hashtable) (VisuAlgo (National University of Singapore), 20 min): Insira algumas chaves com Separate Chaining, depois com Linear Probing, e veja para onde vão as colisões. Repare no fator de carga α = N/M e por que a tabela é refeita maior quando α cresce.
- [Prefix Sums](https://usaco.guide/silver/prefix-sums) (USACO Guide, 10 min): O array de prefixos e a fórmula da soma de um intervalo, prefix[r + 1] − prefix[l]. Budget Periods acrescenta um passo: um mapa das somas de prefixo vistas até agora. Escolha Java nas configurações do site para ver código em Java.
- [Using Maps to Store Key Value Pairs](https://dev.java/learn/api/collections-and-streams/collections-framework/maps/) (dev.java (Oracle), 7 min): Como o HashMap guarda pares, as operações para obter, conferir, atualizar e remover, como percorrer as entradas, e por que uma chave não pode mudar enquanto está num mapa.

## Verifique

1. Por que um HashMap continua rápido depois de um milhão de inserções?
2. Que chave coloca "listen", "silent" e "enlist" no mesmo grupo?
3. Em `3 4 -1 2`, qual intervalo soma 6?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Grupos de anagramas** · Fácil
- **Maior sequência** · Médio
- **Períodos do orçamento** · Médio
- **Tabela hash** · Difícil
- Teste: **Relatório de logs do servidor** · Difícil